imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

from ClassificationModel.src.models.classification_model import CancerClassificationModel
from ClassificationModel.src.utils.dataset_utils import load_dataset
from constants.classification.datasets_constants import DatasetConstants
from constants.classification.model_constants import ModelConstants
from utils.callbacks.notification_callback import NotificationCallback
from utils.class_weight_utils import calculate_class_weights
from tensorflow import keras
from ClassificationModel.src.data_processing.image_augmentation import ImageAugmentationPipeline

Loading the dataset and creating the model

In [ ]:
dataset = load_dataset()

CHECKPOINT = ModelConstants.CHECKPOINT_FILE_PATH

model = CancerClassificationModel(dataset,
                                  (*DatasetConstants.IMAGE_SIZE, DatasetConstants.CHANNELS)
                                  ,checkpoint_path=CHECKPOINT)

augmenter = ImageAugmentationPipeline()

EPOCHS=50

defining callbacks

In [ ]:
callbacks = [
    NotificationCallback(
        notifier=model.notifier,
        metrics_to_track=[
            ModelConstants.TRACKED_ACCURACY_METRIC,
            ModelConstants.TRACKED_LOSS_METRIC,
            ModelConstants.TRACKED_VAL_ACCURACY_METRIC,
            ModelConstants.TRACKED_VAL_LOSS_METRIC
        ]),
    keras.callbacks.ModelCheckpoint(
        filepath=ModelConstants.CHECKPOINT_FILE_PATH,
        save_best_only=True,
        monitor=ModelConstants.TRACKED_VAL_LOSS_METRIC
    ),
    keras.callbacks.EarlyStopping(
        monitor=ModelConstants.TRACKED_VAL_LOSS_METRIC,
        patience=15, 
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor=ModelConstants.TRACKED_VAL_LOSS_METRIC,
        factor=0.5,
        patience=5,  
        min_lr=1e-7
    )
]

In [ ]:
# Calculate class weights to handle imbalance
class_weights = calculate_class_weights(
    dataset[DatasetConstants.TRAIN_SPLIT_NAME],
    class_names=dataset[DatasetConstants.CLASS_NAMES_KEY]
)

training the model

In [ ]:
history = model.train_model(
    epochs=EPOCHS, 
    callbacks=callbacks, 
    augment_train=True, 
    augmenter=augmenter,
    class_weight=class_weights  # Handle class imbalance
)

evaluating the model

In [ ]:

results = model.evaluate_model(present_metrics=True,send_message=True)